In [18]:
#imports and data display structure things
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)


In [ ]:
#PMJDY DATA
pmjdy_path = r"C:\Users\Mihika\Desktop\dsm_project_final\dsm project\data\raw\PMJDY.csv"
pmjdy_raw = pd.read_csv(pmjdy_path)
print("Shape:", pmjdy_raw.shape)
print("Columns:", pmjdy_raw.columns.tolist())
print("First 5 rows:")
pmjdy_raw.head(5)
# not using for now need state wise 

Shape: (36, 10)
Columns: ['Country', 'Year', 'As On Date', 'Name Of The Bank', 'Types Of Banks', 'Accounts Opened In Rural Areas Under The Pradhan Mantri Jan Dhan Yojana (Pmjdy) By Various Banks (UOM:Number), Scaling Factor:1', 'Accounts Opened In Urban Areas Under The Pradhan Mantri Jan Dhan Yojana (Pmjdy) By Different Banks (UOM:Number), Scaling Factor:1', 'Number Of Beneficiaries (UOM:Number), Scaling Factor:1', 'The Cumulative Balance In Accounts (UOM:INR(IndianRupees)), Scaling Factor:100000', 'Rupay Debit Cards Issued By Each Bank (UOM:Number), Scaling Factor:1']
First 5 rows:


,Country,Year,As On Date,Name Of The Bank,Types Of Banks,"Accounts Opened In Rural Areas Under The Pradhan Mantri Jan Dhan Yojana (Pmjdy) By Various Banks (UOM:Number), Scaling Factor:1","Accounts Opened In Urban Areas Under The Pradhan Mantri Jan Dhan Yojana (Pmjdy) By Different Banks (UOM:Number), Scaling Factor:1","Number Of Beneficiaries (UOM:Number), Scaling Factor:1","The Cumulative Balance In Accounts (UOM:INR(IndianRupees)), Scaling Factor:100000","Rupay Debit Cards Issued By Each Bank (UOM:Number), Scaling Factor:1"
0,India,"Financial Year (Apr - Mar), 2026",11-Mar-2026,Axis Bank Ltd,Major Private Banks,206000,1248758,1454758,117905.63,908394
1,India,"Financial Year (Apr - Mar), 2026",11-Mar-2026,Bank of Baroda,Public Sector Banks,46053749,20196760,66250509,4150103.01,61287584
2,India,"Financial Year (Apr - Mar), 2026",11-Mar-2026,Bank of Baroda,Regional Rural Bank,21339242,3406968,24746210,1668228.11,10618719
3,India,"Financial Year (Apr - Mar), 2026",11-Mar-2026,Bank of India,Public Sector Banks,24247756,5271969,29519725,1773069.09,25902292
4,India,"Financial Year (Apr - Mar), 2026",11-Mar-2026,Bank of India,Regional Rural Bank,6257995,866576,7124571,207667.99,3391932


In [ ]:
#PMJDY statewise

pmjdy = pd.read_csv(r"C:\Users\Mihika\Desktop\dsm_project_final\dsm project\data\raw\PMJDY_statewise.csv")  


# Renamed columns to standard names
pmjdy.columns = [
    'sno', 'state_name', 'rural_accounts', 'urban_accounts',
    'total_accounts', 'total_balance_crore', 'rupay_cards'
]

#dropping last total row cause we dont need for statewise analysis
pmjdy = pmjdy[pmjdy['state_name'] != 'Total'].copy()

num_cols = ['rural_accounts', 'urban_accounts', 'total_accounts', 
            'total_balance_crore', 'rupay_cards']

for col in num_cols:
    pmjdy[col] = pd.to_numeric(
        pmjdy[col].astype(str).str.replace(',', '').str.strip(),
        errors='coerce'
    )


# Convert balance from crore to actual rupees
pmjdy['total_balance'] = pmjdy['total_balance_crore'] * 1e7

# Clean numeric columns
num_cols = ['rural_accounts', 'urban_accounts', 'total_accounts', 
            'total_balance', 'rupay_cards']
for col in num_cols:
    pmjdy[col] = pd.to_numeric(
        pmjdy[col].astype(str).str.replace(',', '').str.strip(),
        errors='coerce'
    )


# Derived variables
pmjdy['avg_balance_per_account'] = pmjdy['total_balance'] / pmjdy['total_accounts']
pmjdy['rupay_activation_ratio']  = pmjdy['rupay_cards']   / pmjdy['total_accounts']
pmjdy['rural_share']             = pmjdy['rural_accounts'] / pmjdy['total_accounts']
pmjdy['urban_share']             = pmjdy['urban_accounts'] / pmjdy['total_accounts']

print("Shape:", pmjdy.shape)
print("States:", pmjdy['state_name'].tolist())
pmjdy.head()
pmjdy.to_csv(r"C:\Users\Mihika\Desktop\dsm_project_final\dsm project\data\cleaned\PMJDY_state.csv", index=False)

Shape: (36, 12)
States: ['Andaman And Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu And Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Lakshadweep', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'The Dadra And Nagar Haveli And Daman And Diu', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']


In [22]:
#CENSUS DATA

census_path = r"C:\Users\Mihika\Desktop\dsm_project_final\dsm project\data\raw\census_2011.csv"

census_raw = pd.read_csv(census_path)


print("Shape:", census_raw.shape)
print("Columns:", census_raw.columns.tolist())
census_raw.head(3)

Shape: (606573, 92)
Columns: ['Country', 'State', 'District', 'Sub-District', 'Village Or Town Name', 'Year', 'Residence Type', 'Households (UOM:Number), Scaling Factor:1', 'Population (UOM:Number), Scaling Factor:1', 'Male Population (UOM:Number), Scaling Factor:1', 'Female Population (UOM:Number), Scaling Factor:1', 'Population In The Age Group 0 To 6 Years (UOM:Number), Scaling Factor:1', 'Male Population In The Age Group 0 To 6 Years (UOM:Number), Scaling Factor:1', 'Female Population In The Age Group 0 To 6 Years (UOM:Number), Scaling Factor:1', 'Scheduled Caste Population (UOM:Number), Scaling Factor:1', 'Male Scheduled Caste Population (UOM:Number), Scaling Factor:1', 'Female Scheduled Caste Population  (UOM:Number), Scaling Factor:1', 'Scheduled Tribe Population (UOM:Number), Scaling Factor:1', 'Male Scheduled Tribe Population (UOM:Number), Scaling Factor:1', 'Female Scheduled Tribe Population  (UOM:Number), Scaling Factor:1', 'Literate Population  (UOM:Number), Scaling Factor:

,Country,State,District,Sub-District,Village Or Town Name,Year,Residence Type,"Households (UOM:Number), Scaling Factor:1","Population (UOM:Number), Scaling Factor:1","Male Population (UOM:Number), Scaling Factor:1","Female Population (UOM:Number), Scaling Factor:1","Population In The Age Group 0 To 6 Years (UOM:Number), Scaling Factor:1","Male Population In The Age Group 0 To 6 Years (UOM:Number), Scaling Factor:1","Female Population In The Age Group 0 To 6 Years (UOM:Number), Scaling Factor:1","Scheduled Caste Population (UOM:Number), Scaling Factor:1","Male Scheduled Caste Population (UOM:Number), Scaling Factor:1","Female Scheduled Caste Population (UOM:Number), Scaling Factor:1","Scheduled Tribe Population (UOM:Number), Scaling Factor:1","Male Scheduled Tribe Population (UOM:Number), Scaling Factor:1","Female Scheduled Tribe Population (UOM:Number), Scaling Factor:1","Literate Population (UOM:Number), Scaling Factor:1","Male Literate Population (UOM:Number), Scaling Factor:1","Female Literate Population (UOM:Number), Scaling Factor:1","Illiterate Population (UOM:Number), Scaling Factor:1","Male Illiterate Population (UOM:Number), Scaling Factor:1","Female Illiterate Population (UOM:Number), Scaling Factor:1","Working Population (UOM:Number), Scaling Factor:1","Male Working Population (UOM:Number), Scaling Factor:1","Female Woking Population (UOM:Number), Scaling Factor:1","Number Of Main Workers (UOM:Number), Scaling Factor:1","Number Of Male Main Workers (UOM:Number), Scaling Factor:1","Number Of Female Main Worker (UOM:Number), Scaling Factor:1","Number Of Main Workers As Cultivators (UOM:Number), Scaling Factor:1","Number Of Male Main Workers As Cultivators (UOM:Number), Scaling Factor:1","Number Of Female Main Workers As Cultivators (UOM:Number), Scaling Factor:1","Number Of Main Workers As Agricultural Labourers (UOM:Number), Scaling Factor:1","Number Of Male Main Workers As Agricultural Labourers (UOM:Number), Scaling Factor:1","Number Of Female Main Workers As Agricultural Labourers (UOM:Number), Scaling Factor:1","Number Of Main Workers In The Household Industry (UOM:Number), Scaling Factor:1","Number Of Male Main Workers In The Household Industry (UOM:Number), Scaling Factor:1","Number Of Female Main Workers In The Household Industry (UOM:Number), Scaling Factor:1","Number Of Main Workers As Other Workers (UOM:Number), Scaling Factor:1","Number Of Male Main Workers As Other Workers (UOM:Number), Scaling Factor:1","Number Of Female Main Workers As Other Workers (UOM:Number), Scaling Factor:1","Number Of Marginal Workers (UOM:Number), Scaling Factor:1","Number Of Male Marginal Workers (UOM:Number), Scaling Factor:1","Number Of Female Marginal Workers (UOM:Number), Scaling Factor:1","Number Of Marginal Workers As Cultivators (UOM:Number), Scaling Factor:1","Number Of Male Marginal Workers As Cultivators (UOM:Number), Scaling Factor:1","Number Of Female Marginal Workers As Cultivators (UOM:Number), Scaling Factor:1","Number Of Marginal Workers As Agricultural Labourers (UOM:Number), Scaling Factor:1","Number Of Male Marginal Workers As Agricultural Labourers (UOM:Number), Scaling Factor:1","Number Of Female Marginal Workers As Agricultural Labourers (UOM:Number), Scaling Factor:1","Number Of Marginal Workers In The Household Industry (UOM:Number), Scaling Factor:1","Number Of Male Marginal Workers In The Household Industry (UOM:Number), Scaling Factor:1","Number Of Female Marginal Workers In The Household Industry (UOM:Number), Scaling Factor:1","Number Of Marginal Workers As Other Workers (UOM:Number), Scaling Factor:1","Number Of Male Marginal Workers As Other Workers (UOM:Number), Scaling Factor:1","Number Of Female Marginal Workers As Other Workers (UOM:Number), Scaling Factor:1","Number Of Marginal Workers Who Worked For 3 To 6 Months (UOM:Number), Scaling Factor:1","Number Of Male Marginal Workers Who Worked For 3 To 6 Months (UOM:Number), Scaling Factor:1","Number Of Female Marginal Workers Who Worked For 3 To 

In [23]:

# renamed columns without extra spaces
census_raw = census_raw.rename(columns={
    'State':                                                         'state_name',
    'Residence Type':                                                'residence_type',
    'Households (UOM:Number), Scaling Factor:1':                     'households',
    'Population (UOM:Number), Scaling Factor:1':                     'population',
    'Male Population (UOM:Number), Scaling Factor:1':                'male_pop',
    'Female Population (UOM:Number), Scaling Factor:1':              'female_pop',
    'Scheduled Caste Population (UOM:Number), Scaling Factor:1':     'sc_population',
    'Scheduled Tribe Population (UOM:Number), Scaling Factor:1':     'st_population',
    'Male Literate Population (UOM:Number), Scaling Factor:1':       'male_literate',
    'Working Population (UOM:Number), Scaling Factor:1':             'working_pop',
    'Female Woking Population (UOM:Number), Scaling Factor:1':       'female_working_pop',
    'Number Of Non Workers (UOM:Number), Scaling Factor:1':          'non_workers',
})

# rename columns with extra spaces 
census_raw = census_raw.rename(columns={
    'Literate Population  (UOM:Number), Scaling Factor:1':            'literate_pop',
    'Female Literate Population  (UOM:Number), Scaling Factor:1':     'female_literate',
})

#  filtered only needed columns
keep_cols = ['state_name', 'residence_type', 'households', 'population',
             'male_pop', 'female_pop', 'literate_pop', 'male_literate',
             'female_literate', 'sc_population', 'st_population',
             'working_pop', 'female_working_pop']

census = census_raw[keep_cols].copy()

# converted to numeric
num_cols = ['households', 'population', 'male_pop', 'female_pop',
            'literate_pop', 'male_literate', 'female_literate',
            'sc_population', 'st_population', 'working_pop', 'female_working_pop']
for col in num_cols:
    census[col] = pd.to_numeric(census[col], errors='coerce')



# aggregate from village to state level
census_state = census.groupby('state_name')[num_cols].sum().reset_index()

# derived variables
census_state['literacy_rate']        = (census_state['literate_pop']       / census_state['population']) * 100
census_state['male_literacy_rate']   = (census_state['male_literate']      / census_state['male_pop'])   * 100
census_state['female_literacy_rate'] = (census_state['female_literate']    / census_state['female_pop']) * 100
census_state['female_workforce_pct'] = (census_state['female_working_pop'] / census_state['female_pop']) * 100
census_state['sc_share']             = (census_state['sc_population']      / census_state['population']) * 100
census_state['st_share']             = (census_state['st_population']      / census_state['population']) * 100
census_state['avg_household_size']   = census_state['population']          / census_state['households']

# urban/rural split saved separately 
census_rural_urban = census.groupby(
    ['state_name', 'residence_type'])[num_cols].sum().reset_index()

census_state.to_csv(r"C:\Users\Mihika\Desktop\dsm_project_final\dsm project\data\cleaned/census_state.csv", index=False)
census_rural_urban.to_csv(r"C:\Users\Mihika\Desktop\dsm_project_final\dsm project\data\cleaned/census_rural_urban.csv", index=False)



In [24]:
# NFHS DATA
nfhs_path = r"C:\Users\Mihika\Desktop\dsm_project_final\dsm project\data\raw\NFHS.csv"
nfhs_raw = pd.read_csv(nfhs_path)
print("Shape:", nfhs_raw.shape)
print("Columns:", nfhs_raw.columns.tolist())
nfhs_raw.head(3)

Shape: (108, 136)
Columns: ['Country', 'State', 'Year', 'Residence Type', 'Nfhs Survey Number (Nfhs - 4 Or Nfhs - 5)', 'Women Having A Bank Or Savings Account That They Themselves Use (%) (UOM:%(Percentage)), Scaling Factor:1', 'Adolescent Fertility Rate For Women Age 15 To 19 Years (%) (UOM:%(Percentage)), Scaling Factor:1', 'Women Age Group 15 To 19 Years Who Are Anaemic (%) (UOM:%(Percentage)), Scaling Factor:1', 'Women Age Group 15 To 49 Years Who Are Anaemic (%) (UOM:%(Percentage)), Scaling Factor:1', 'Married Women Age Group 15 To 49 Years Use Any Family Planning Methods (%) (UOM:%(Percentage)), Scaling Factor:1', 'Married Women Age Group 15 To 49 Years Any Modern Family Planning Method (%) (UOM:%(Percentage)), Scaling Factor:1', 'Average Out Of Pocket Expenditure For Each Delivery In Public Health Facility (UOM:INR(IndianRupees)), Scaling Factor:1', 'Births Attended By Skilled Health Personnel (%) (UOM:%(Percentage)), Scaling Factor:1', 'Births Delivered By Caesarean Section (%)

,Country,State,Year,Residence Type,Nfhs Survey Number (Nfhs - 4 Or Nfhs - 5),"Women Having A Bank Or Savings Account That They Themselves Use (%) (UOM:%(Percentage)), Scaling Factor:1","Adolescent Fertility Rate For Women Age 15 To 19 Years (%) (UOM:%(Percentage)), Scaling Factor:1","Women Age Group 15 To 19 Years Who Are Anaemic (%) (UOM:%(Percentage)), Scaling Factor:1","Women Age Group 15 To 49 Years Who Are Anaemic (%) (UOM:%(Percentage)), Scaling Factor:1","Married Women Age Group 15 To 49 Years Use Any Family Planning Methods (%) (UOM:%(Percentage)), Scaling Factor:1","Married Women Age Group 15 To 49 Years Any Modern Family Planning Method (%) (UOM:%(Percentage)), Scaling Factor:1","Average Out Of Pocket Expenditure For Each Delivery In Public Health Facility (UOM:INR(IndianRupees)), Scaling Factor:1","Births Attended By Skilled Health Personnel (%) (UOM:%(Percentage)), Scaling Factor:1","Births Delivered By Caesarean Section (%) (UOM:%(Percentage)), Scaling Factor:1","Births In Private Health Facility Delivered By Caesarean Section (%) (UOM:%(Percentage)), Scaling Factor:1","Births In Public Health Facility Delivered By Caesarean Section (%) (UOM:%(Percentage)), Scaling Factor:1","Breastfeeding Children Age Group 6 To 23 Months Receiving An Adequate Diet (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 12 To 23 Months Fully Vaccinated Based On Information From Either Vaccination Card Or Mothers Recall (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 12 To 23 Months Fully Vaccinated Based On Information From Vaccination Card Only (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 12 To 23 Months Who Have Received 3 Doses Of Penta Or Diphtheria Tetanus Toxoids And Pertussis (Dtp) Vaccine (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 12 To 23 Months Who Have Received 3 Doses Of Penta Or Hepatitis B Vaccine (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 12 To 23 Months Who Have Received 3 Doses Of Polio Vaccine (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 12 To 23 Months Who Have Received 3 Doses Of Rotavirus Vaccine (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 12 To 23 Months Who Have Received Bacillus Calmette Guerin (Bcg) (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 12 To 23 Months Who Have Received The First Dose Of Measles Containing Vaccine (Mcv) (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 12 To 23 Months Who Received Most Of Their Vaccinations In A Private Health Facility (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 12 To 23 Months Who Received Most Of Their Vaccinations In A Public Health Facility (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 24 To 35 Months Who Have Received A Second Dose Of Measles Containing Vaccine (Mcv) (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 5 Years Who Attended Pre-Primary School During The Year 2019-20 (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 6 To 59 Months Who Are Anaemic (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 6 To 8 Months Receiving Solid Or Semisolid Food And Breastmilk (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 9 To 35 Months Who Received Vitamin A Dose In The Last 6 Months (%) (UOM:%(Percentage)), Scaling Factor:1","Children Born At Home Who Were Taken To A Health Facility For A Check-Up Within 24 Hours Of Birth (%) (UOM:%(Percentage)), Scaling Factor:1","Children Under 5 Years Who Are Overweight (Weight-For-Height) (%) (UOM:%(Percentage)), Scaling Factor:1","Children Under 5 Years Who Are Severely Wasted (Weight-For-Height) (%) (UOM:%(Percentage)), Scaling Factor:1","Children Under 5 Years Who Are Stunted (Height-For-Age) (%) (UOM:%(Percentage)), Scaling Factor:1","Children Under 5 Years Who Are Underweight (Weight-For-Age) (%) (UOM:%(Percentage)), Scaling Factor:1","Children Under 5 Years Who Are Wasted (Weight-For-Hei

In [25]:

NFHS_COLS = {
    'state_col':                'State',
    'women_with_bank_account':  'Women_Bank_Account_%',   
    'men_with_bank_account':    'Men_Bank_Account_%',     
    'female_literacy':          'Female_Literacy_%',
    'wealth_index_poorest':     'Wealth_Poorest_%',
    'female_workforce':         'Female_Workforce_%',
}

nfhs = nfhs_raw.rename(columns={v: k for k, v in NFHS_COLS.items()})
nfhs = nfhs.rename(columns={'state_col': 'state_name'})

print(nfhs['Nfhs Survey Number (Nfhs - 4 Or Nfhs - 5)'].unique())
print(nfhs['Residence Type'].unique())

num_cols = list(NFHS_COLS.keys())[1:]   
for col in num_cols:
    if col in nfhs.columns:
        nfhs[col] = pd.to_numeric(
            nfhs[col].astype(str).str.replace('%','').str.replace(',','').str.strip(),
            errors='coerce'
        )

# gender gap 
if 'men_with_bank_account' in nfhs.columns and 'women_with_bank_account' in nfhs.columns:
    nfhs['gender_gap'] = nfhs['men_with_bank_account'] - nfhs['women_with_bank_account']

print("Shape:", nfhs.shape)
nfhs.head()

['NFHS-5' 'NFHS-4']
['Rural' 'Urban' 'Total']
Shape: (108, 136)


,Country,state_name,Year,Residence Type,Nfhs Survey Number (Nfhs - 4 Or Nfhs - 5),"Women Having A Bank Or Savings Account That They Themselves Use (%) (UOM:%(Percentage)), Scaling Factor:1","Adolescent Fertility Rate For Women Age 15 To 19 Years (%) (UOM:%(Percentage)), Scaling Factor:1","Women Age Group 15 To 19 Years Who Are Anaemic (%) (UOM:%(Percentage)), Scaling Factor:1","Women Age Group 15 To 49 Years Who Are Anaemic (%) (UOM:%(Percentage)), Scaling Factor:1","Married Women Age Group 15 To 49 Years Use Any Family Planning Methods (%) (UOM:%(Percentage)), Scaling Factor:1","Married Women Age Group 15 To 49 Years Any Modern Family Planning Method (%) (UOM:%(Percentage)), Scaling Factor:1","Average Out Of Pocket Expenditure For Each Delivery In Public Health Facility (UOM:INR(IndianRupees)), Scaling Factor:1","Births Attended By Skilled Health Personnel (%) (UOM:%(Percentage)), Scaling Factor:1","Births Delivered By Caesarean Section (%) (UOM:%(Percentage)), Scaling Factor:1","Births In Private Health Facility Delivered By Caesarean Section (%) (UOM:%(Percentage)), Scaling Factor:1","Births In Public Health Facility Delivered By Caesarean Section (%) (UOM:%(Percentage)), Scaling Factor:1","Breastfeeding Children Age Group 6 To 23 Months Receiving An Adequate Diet (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 12 To 23 Months Fully Vaccinated Based On Information From Either Vaccination Card Or Mothers Recall (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 12 To 23 Months Fully Vaccinated Based On Information From Vaccination Card Only (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 12 To 23 Months Who Have Received 3 Doses Of Penta Or Diphtheria Tetanus Toxoids And Pertussis (Dtp) Vaccine (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 12 To 23 Months Who Have Received 3 Doses Of Penta Or Hepatitis B Vaccine (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 12 To 23 Months Who Have Received 3 Doses Of Polio Vaccine (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 12 To 23 Months Who Have Received 3 Doses Of Rotavirus Vaccine (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 12 To 23 Months Who Have Received Bacillus Calmette Guerin (Bcg) (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 12 To 23 Months Who Have Received The First Dose Of Measles Containing Vaccine (Mcv) (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 12 To 23 Months Who Received Most Of Their Vaccinations In A Private Health Facility (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 12 To 23 Months Who Received Most Of Their Vaccinations In A Public Health Facility (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 24 To 35 Months Who Have Received A Second Dose Of Measles Containing Vaccine (Mcv) (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 5 Years Who Attended Pre-Primary School During The Year 2019-20 (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 6 To 59 Months Who Are Anaemic (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 6 To 8 Months Receiving Solid Or Semisolid Food And Breastmilk (%) (UOM:%(Percentage)), Scaling Factor:1","Children Age Group 9 To 35 Months Who Received Vitamin A Dose In The Last 6 Months (%) (UOM:%(Percentage)), Scaling Factor:1","Children Born At Home Who Were Taken To A Health Facility For A Check-Up Within 24 Hours Of Birth (%) (UOM:%(Percentage)), Scaling Factor:1","Children Under 5 Years Who Are Overweight (Weight-For-Height) (%) (UOM:%(Percentage)), Scaling Factor:1","Children Under 5 Years Who Are Severely Wasted (Weight-For-Height) (%) (UOM:%(Percentage)), Scaling Factor:1","Children Under 5 Years Who Are Stunted (Height-For-Age) (%) (UOM:%(Percentage)), Scaling Factor:1","Children Under 5 Years Who Are Underweight (Weight-For-Age) (%) (UOM:%(Percentage)), Scaling Factor:1","Children Under 5 Years Who Are Wasted (Weight-Fo

In [26]:
# filter to NFHS-5 only
nfhs_5 = nfhs[nfhs['Nfhs Survey Number (Nfhs - 4 Or Nfhs - 5)'] == 'NFHS-5'].copy()

#  rename the columns before aggregating
nfhs_5 = nfhs_5.rename(columns={
    'Women Having A Bank Or Savings Account That They Themselves Use (%) (UOM:%(Percentage)), Scaling Factor:1': 'women_bank_account_pct',
    'Women Age Group 15 To 49 Years Who Are Literate (%) (UOM:%(Percentage)), Scaling Factor:1':               'female_literacy_nfhs',
    'Men Age Group 15 To 49 Years Who Are Literate (%) (UOM:%(Percentage)), Scaling Factor:1':                 'male_literacy_nfhs',
    'Women Who Worked In The Last 12 Months And Were Paid In Cash (%) (UOM:%(Percentage)), Scaling Factor:1':  'female_workforce_pct_nfhs',
    'Women Having A Mobile Phone That They Themselves Use (%) (UOM:%(Percentage)), Scaling Factor:1':          'women_mobile_pct',
    'Population Living In Households With Electricity (%) (UOM:%(Percentage)), Scaling Factor:1':              'electricity_access_pct',
    'Population Living In Households With An Improved Drinking Water Source (%) (UOM:%(Percentage)), Scaling Factor:1': 'clean_water_pct',
    'Women Owning A House And Or Land (Alone Or Jointly With Others) (%) (UOM:%(Percentage)), Scaling Factor:1': 'women_property_ownership_pct',
})

#  convert to numeric
num_cols_nfhs = ['women_bank_account_pct', 'female_literacy_nfhs',
                 'male_literacy_nfhs', 'female_workforce_pct_nfhs',
                 'women_mobile_pct', 'electricity_access_pct',
                 'clean_water_pct', 'women_property_ownership_pct']

for col in num_cols_nfhs:
    nfhs_5[col] = pd.to_numeric(nfhs_5[col], errors='coerce')

#  aggregate Rural + Urban → state total by averaging
nfhs_clean = nfhs_5.groupby('state_name')[num_cols_nfhs].mean().reset_index()

#  gender gap
nfhs_clean['literacy_gender_gap'] = nfhs_clean['male_literacy_nfhs'] - nfhs_clean['female_literacy_nfhs']
nfhs_clean.to_csv(r"C:\Users\Mihika\Desktop\dsm_project_final\dsm project\data\cleaned\nfhs_clean.csv", index=False)

print("NFHS cleaned shape:", nfhs_clean.shape)
print("States:", nfhs_clean['state_name'].tolist())
nfhs_clean.head()

NFHS cleaned shape: (36, 10)
States: ['Andaman and Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu and Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Lakshadweep', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'The Dadra and Nagar Haveli and Daman and Diu', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']


,state_name,women_bank_account_pct,female_literacy_nfhs,male_literacy_nfhs,female_workforce_pct_nfhs,women_mobile_pct,electricity_access_pct,clean_water_pct,women_property_ownership_pct,literacy_gender_gap
0,Andaman and Nicobar Islands,89.15,86.10,92.00,25.90,80.85,98.00,96.65,15.15,5.90
1,Andhra Pradesh,83.15,71.40,81.35,40.50,54.15,99.50,97.40,46.00,9.95
2,Arunachal Pradesh,81.35,75.95,87.95,23.30,78.60,96.75,95.50,70.05,12.00
3,Assam,79.90,81.45,87.70,18.25,64.65,95.25,88.65,40.10,6.25
4,Bihar,77.65,64.70,80.50,12.25,55.55,96.25,99.35,54.55,15.80


In [27]:
nfhs_clean['state_name'] = nfhs_clean['state_name'].str.title()

In [28]:
#RBI HANDBOOK 
rbi_path = r"C:\Users\Mihika\Desktop\dsm_project_final\dsm project\data\raw\Handbook.csv"
rbi_raw = pd.read_csv(rbi_path)
print("Shape:", rbi_raw.shape)
print("Columns:", rbi_raw.columns.tolist())
rbi_raw.head(3)

Shape: (813, 17)
Columns: ['Country', 'State', 'Year', 'Name Of The Region', 'Additional Info', 'Distribution Of Offices Of Scheduled Commercial Banks (UOM:Number), Scaling Factor:1', 'Credit-Deposit Ratio Of Scheduled Commercial Banks According To Place Of Sanction (UOM:Ratio), Scaling Factor:1', 'Credit-Deposit Ratio Of Scheduled Commercial Banks According To Place Of Utilisation (UOM:Ratio), Scaling Factor:1', 'Deposits By Scheduled Commercial Banks In India (UOM:INR(IndianRupees)), Scaling Factor:10000000', 'Credit By Scheduled Commercial Banks In India (UOM:INR(IndianRupees)), Scaling Factor:10000000', 'Credit To Agriculture By Scheduled Commercial Banks (UOM:INR(IndianRupees)), Scaling Factor:10000000', 'Credit To Industry By Scheduled Commercial Banks (UOM:INR(IndianRupees)), Scaling Factor:10000000', 'Personal Loans By Scheduled Commercial Banks (UOM:INR(IndianRupees)), Scaling Factor:10000000', 'Deposits Of Regional Rural Banks (UOM:INR(IndianRupees)), Scaling Factor:10000000'

,Country,State,Year,Name Of The Region,Additional Info,"Distribution Of Offices Of Scheduled Commercial Banks (UOM:Number), Scaling Factor:1","Credit-Deposit Ratio Of Scheduled Commercial Banks According To Place Of Sanction (UOM:Ratio), Scaling Factor:1","Credit-Deposit Ratio Of Scheduled Commercial Banks According To Place Of Utilisation (UOM:Ratio), Scaling Factor:1","Deposits By Scheduled Commercial Banks In India (UOM:INR(IndianRupees)), Scaling Factor:10000000","Credit By Scheduled Commercial Banks In India (UOM:INR(IndianRupees)), Scaling Factor:10000000","Credit To Agriculture By Scheduled Commercial Banks (UOM:INR(IndianRupees)), Scaling Factor:10000000","Credit To Industry By Scheduled Commercial Banks (UOM:INR(IndianRupees)), Scaling Factor:10000000","Personal Loans By Scheduled Commercial Banks (UOM:INR(IndianRupees)), Scaling Factor:10000000","Deposits Of Regional Rural Banks (UOM:INR(IndianRupees)), Scaling Factor:10000000","Credit Of Regional Rural Banks (UOM:INR(IndianRupees)), Scaling Factor:10000000","Credit-Deposit Ratio Of Regional Rural Banks (UOM:Ratio), Scaling Factor:1","Number Of Branches Of Regional Rural Banks (UOM:Number), Scaling Factor:1"
0,India,Andaman and Nicobar Islands,"Calendar Year (Jan - Dec), 2025",Eastern Region,NaN,76.0,52.2,52.5,8549.0,4485.0,365.0,155.0,2975.0,NaN,NaN,NaN,NaN
1,India,Andhra Pradesh,"Calendar Year (Jan - Dec), 2025",Southern Region,NaN,8087.0,151.1,157.2,529673.0,832527.0,282319.0,89275.0,290337.0,52883.0,62573.0,118.3,1367.0
2,India,Arunachal Pradesh,"Calendar Year (Jan - Dec), 2025",North-Eastern Region,NaN,229.0,31.0,32.2,33114.0,10653.0,565.0,1021.0,7222.0,1511.0,515.0,34.1,34.0


In [29]:
RBI_COLS = {
    'state_name':           'State',
    'total_bank_branches': 'Total_Bank_Branches',
    'rural_branches':      'Rural_Branches',
    'urban_branches':      'Urban_Branches',
    'credit_deposit_ratio':'CD_Ratio',
}

rbi = rbi_raw.rename(columns={v: k for k, v in RBI_COLS.items()})


num_cols = ['total_bank_branches', 'rural_branches', 'urban_branches', 'credit_deposit_ratio']
for col in num_cols:
    if col in rbi.columns:
        rbi[col] = pd.to_numeric(
            rbi[col].astype(str).str.replace(',','').str.strip(),
            errors='coerce'
        )

rbi.to_csv(r"C:\Users\Mihika\Desktop\dsm_project_final\dsm project\data\cleanedrbi.csv", index=False)
print("Shape:", rbi.shape)
rbi.head()

Shape: (813, 17)


,Country,state_name,Year,Name Of The Region,Additional Info,"Distribution Of Offices Of Scheduled Commercial Banks (UOM:Number), Scaling Factor:1","Credit-Deposit Ratio Of Scheduled Commercial Banks According To Place Of Sanction (UOM:Ratio), Scaling Factor:1","Credit-Deposit Ratio Of Scheduled Commercial Banks According To Place Of Utilisation (UOM:Ratio), Scaling Factor:1","Deposits By Scheduled Commercial Banks In India (UOM:INR(IndianRupees)), Scaling Factor:10000000","Credit By Scheduled Commercial Banks In India (UOM:INR(IndianRupees)), Scaling Factor:10000000","Credit To Agriculture By Scheduled Commercial Banks (UOM:INR(IndianRupees)), Scaling Factor:10000000","Credit To Industry By Scheduled Commercial Banks (UOM:INR(IndianRupees)), Scaling Factor:10000000","Personal Loans By Scheduled Commercial Banks (UOM:INR(IndianRupees)), Scaling Factor:10000000","Deposits Of Regional Rural Banks (UOM:INR(IndianRupees)), Scaling Factor:10000000","Credit Of Regional Rural Banks (UOM:INR(IndianRupees)), Scaling Factor:10000000","Credit-Deposit Ratio Of Regional Rural Banks (UOM:Ratio), Scaling Factor:1","Number Of Branches Of Regional Rural Banks (UOM:Number), Scaling Factor:1"
0,India,Andaman and Nicobar Islands,"Calendar Year (Jan - Dec), 2025",Eastern Region,NaN,76.0,52.2,52.5,8549.0,4485.0,365.0,155.0,2975.0,NaN,NaN,NaN,NaN
1,India,Andhra Pradesh,"Calendar Year (Jan - Dec), 2025",Southern Region,NaN,8087.0,151.1,157.2,529673.0,832527.0,282319.0,89275.0,290337.0,52883.0,62573.0,118.3,1367.0
2,India,Arunachal Pradesh,"Calendar Year (Jan - Dec), 2025",North-Eastern Region,NaN,229.0,31.0,32.2,33114.0,10653.0,565.0,1021.0,7222.0,1511.0,515.0,34.1,34.0
3,India,Assam,"Calendar Year (Jan - Dec), 2025",North-Eastern Region,NaN,3236.0,62.7,66.0,234070.0,154473.0,16325.0,25553.0,74641.0,12705.0,8107.0,63.8,475.0
4,India,Bihar,"Calendar Year (Jan - Dec), 2025",Eastern Region,NaN,8226.0,51.4,53.5,568726.0,304440.0,69932.0,33000.0,116048.0,47805.0,30493.0,63.8,2118.0


In [30]:
for name, df in [('PMJDY', pmjdy), ('Census', census_state), 
                 ('NFHS', nfhs), ('RBI', rbi)]:
    counts = df.groupby('state_name').size()
    max_rows = counts.max()
    print(f"{name}: {max_rows} rows per state (total rows: {len(df)})")

PMJDY: 1 rows per state (total rows: 36)
Census: 1 rows per state (total rows: 36)
NFHS: 3 rows per state (total rows: 108)
RBI: 43 rows per state (total rows: 813)


In [ ]:
# For bri, keeping only latest year per state
rbi_clean = rbi.sort_values('Year', ascending=False).drop_duplicates(subset='state_name', keep='first')


In [36]:
# MERGE
master = pmjdy.merge(census_state, on='state_name', how='left')
master = master.merge(nfhs_clean,  on='state_name', how='left')
master = master.merge(rbi_clean,         on='state_name', how='left')

# cross-dataset derived variables
master['accounts_per_capita'] = master['total_accounts'] / master['population']
if 'total_bank_branches' in master.columns:
    master['branches_per_lakh'] = (master['total_bank_branches'] / master['population']) * 100000

print("Shape:", master.shape)

missing_census = master[master['population'].isna()]['state_name'].tolist()
missing_nfhs   = master[master['women_bank_account_pct'].isna()]['state_name'].tolist()
print("\nStates missing Census data:", missing_census)
print("States missing NFHS data:",   missing_nfhs)

master.to_csv(r"C:\Users\Mihika\Desktop\dsm_project_final\dsm project\data\cleaned\final_dataset.csv", index=False)
master.head()

Shape: (36, 56)

States missing Census data: []
States missing NFHS data: []


,sno,state_name,rural_accounts,urban_accounts,total_accounts,total_balance_crore,rupay_cards,total_balance,avg_balance_per_account,rupay_activation_ratio,rural_share,urban_share,households,population,male_pop,female_pop,literate_pop,male_literate,female_literate,sc_population,st_population,working_pop,female_working_pop,literacy_rate,male_literacy_rate,female_literacy_rate,female_workforce_pct,sc_share,st_share,avg_household_size,women_bank_account_pct,female_literacy_nfhs,male_literacy_nfhs,female_workforce_pct_nfhs,women_mobile_pct,electricity_access_pct,clean_water_pct,women_property_ownership_pct,literacy_gender_gap,Country,Year,Name Of The Region,Additional Info,"Distribution Of Offices Of Scheduled Commercial Banks (UOM:Number), Scaling Factor:1","Credit-Deposit Ratio Of Scheduled Commercial Banks According To Place Of Sanction (UOM:Ratio), Scaling Factor:1","Credit-Deposit Ratio Of Scheduled Commercial Banks According To Place Of Utilisation (UOM:Ratio), Scaling Factor:1","Deposits By Scheduled Commercial Banks In India (UOM:INR(IndianRupees)), Scaling Factor:10000000","Credit By Scheduled Commercial Banks In India (UOM:INR(IndianRupees)), Scaling Factor:10000000","Credit To Agriculture By Scheduled Commercial Banks (UOM:INR(IndianRupees)), Scaling Factor:10000000","Credit To Industry By Scheduled Commercial Banks (UOM:INR(IndianRupees)), Scaling Factor:10000000","Personal Loans By Scheduled Commercial Banks (UOM:INR(IndianRupees)), Scaling Factor:10000000","Deposits Of Regional Rural Banks (UOM:INR(IndianRupees)), Scaling Factor:10000000","Credit Of Regional Rural Banks (UOM:INR(IndianRupees)), Scaling Factor:10000000","Credit-Deposit Ratio Of Regional Rural Banks (UOM:Ratio), Scaling Factor:1","Number Of Branches Of Regional Rural Banks (UOM:Number), Scaling Factor:1",accounts_per_capita
0,1,Andaman And Nicobar Islands,46862,17645,64507,55.44,39096,5.544000e+08,8594.416110,0.606074,0.726464,0.273536,94551.0,380581.0,202871.0,177710.0,294281.0,164377.0,129904.0,0.0,28530.0,152535.0,31646.0,77.324144,81.025381,73.098869,17.807664,0.000000,7.496433,4.025140,89.15,86.10,92.00,25.90,80.85,98.00,96.65,15.15,5.90,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.169496
1,2,Andhra Pradesh,12453142,4607102,17060244,6464.17,10950935,6.464170e+10,3789.025526,0.641898,0.729951,0.270049,12718976.0,49577103.0,24830513.0,24746590.0,29859982.0,16549514.0,13310468.0,8469278.0,2740133.0,23080964.0,8573546.0,60.229380,66.649908,53.787079,34.645363,17.083043,5.527013,3.897885,83.15,71.40,81.35,40.50,54.15,99.50,97.40,46.00,9.95,India,"Calendar Year (Jan - Dec), 2025",Southern Region,NaN,8087.0,151.1,157.2,529673.0,832527.0,282319.0,89275.0,290337.0,52883.0,62573.0,118.3,1367.0,0.344115
2,3,Arunachal Pradesh,462544,11185,473729,301.90,318210,3.019000e+09,6372.841857,0.671713,0.976389,0.023611,270577.0,1383727.0,713912.0,669815.0,766005.0,439868.0,326137.0,0.0,951821.0,587657.0,237384.0,55.358102,61.613756,48.690609,35.440233,0.000000,68.786762,5.113986,81.35,75.95,87.95,23.30,78.60,96.75,95.50,70.05,12.00,India,"Calendar Year (Jan - Dec), 2025",North-Eastern Region,NaN,229.0,31.0,32.2,33114.0,10653.0,565.0,1021.0,7222.0,1511.0,515.0,34.1,34.0,0.342357
3,4,Assam,23230698,2602724,25833422,7838.20,16255078,7.838200e+10,3034.131522,0.629227,0.899250,0.100750,6305524.0,30771805.0,15715019.0,15056786.0,18818397.0,10376587.0,8441810.0,2206114.0,3868306.0,11800354.0,3390636.0,61.154674,66.029745,56.066481,22.518989,7.169271,12.570943,4.880134,79.90,81.45,87.70,18.25,64.65,95.25,88.65,40.10,6.25,India,"Calendar Year (Jan - Dec), 2025",North-Eastern Region,NaN,3236.0,62.7,66.0,234070.0,154473.0,16325.0,25553.0,74641.0,12705.0,8107.0,63.8,475.0,0.839516
4,5,Bihar,59890944,8663361,68554305,30049.94,49353085,3.004994e+11,4383.377528,0.719912,0.873628,0.126372,18913565.0,104099452.0,54278157.0,49821295.0,52504553.0,31608023.0,20896530.0,16567325.0,1336573.0,34724987.0,9502798.0,50.436916,58.233412,41.942968,19.073768,15.914901,1.283939,

In [ ]:
# negative/impossible values fix
numeric_pct_cols = ['women_bank_account_pct', 'female_literacy_nfhs', 
                    'male_literacy_nfhs', 'electricity_access_pct',
                    'clean_water_pct', 'women_mobile_pct']
for col in numeric_pct_cols:
    if col in master.columns:
        master[col] = master[col].apply(lambda x: np.nan if pd.notna(x) and x < 0 else x)

print("Negative values fixed.")

Negative values fixed.


In [ ]:
print(master.groupby('state_name').size().describe())
# checking merge

count    36.0
mean      1.0
std       0.0
min       1.0
25%       1.0
50%       1.0
75%       1.0
max       1.0
dtype: float64


In [42]:
db_path = r"C:\Users\Mihika\Desktop\dsm_project_final\dsm project\database\pmjdy.db"
import os
os.makedirs(r"C:\Users\Mihika\Desktop\dsm_project_final\dsm project\database", exist_ok=True)

conn = sqlite3.connect(r"C:\Users\Mihika\Desktop\dsm_project_final\dsm project\database\pmjdy.db")

pmjdy.to_sql('pmjdy_accounts',  conn, if_exists='replace', index=False)
census_state.to_sql('census_2011',   conn, if_exists='replace', index=False)
nfhs_clean.to_sql('nfhs_state',     conn, if_exists='replace', index=False)  # updated
rbi.to_sql('rbi_banking',      conn, if_exists='replace', index=False)
master.to_sql('master_dataset', conn, if_exists='replace', index=False)

cols = conn.execute("PRAGMA table_info(nfhs_state);").fetchall()
print("NFHS table columns:", [c[1] for c in cols])


# test join query 

join_query = """
SELECT 
    p.state_name,
    p.total_accounts,
    ROUND(p.rupay_activation_ratio, 3)  AS rupay_ratio,
    ROUND(p.avg_balance_per_account, 2) AS avg_balance,
    ROUND(c.literacy_rate, 1)           AS literacy_rate,
    ROUND(n.women_bank_account_pct, 1)  AS women_bank_pct,
    ROUND(n.literacy_gender_gap, 1)              AS gender_gap
FROM pmjdy_accounts p
LEFT JOIN census_2011  c ON p.state_name = c.state_name
LEFT JOIN nfhs_state   n ON p.state_name = n.state_name
ORDER BY p.rupay_activation_ratio DESC
LIMIT 10;
"""

result = pd.read_sql_query(join_query, conn)
conn.close()

print("\nTop 10 states by RuPay activation ratio:")
print(result.to_string(index=False))

NFHS table columns: ['state_name', 'women_bank_account_pct', 'female_literacy_nfhs', 'male_literacy_nfhs', 'female_workforce_pct_nfhs', 'women_mobile_pct', 'electricity_access_pct', 'clean_water_pct', 'women_property_ownership_pct', 'literacy_gender_gap']

Top 10 states by RuPay activation ratio:
                                  state_name  total_accounts  rupay_ratio  avg_balance  literacy_rate  women_bank_pct  gender_gap
                                       Delhi         7011342        0.777      5474.81           75.0            -6.3       -87.3
                                     Gujarat        19797888        0.764      6347.85           68.0            70.5        13.5
                                    Nagaland          412074        0.758      3746.90           67.9            66.7         7.1
                                 Lakshadweep           10681        0.752     19380.21           81.5            67.0         1.6
                              Madhya Pradesh        